# TrOCR PL — QLoRA fine-tune (Kaggle GPU)

Fine-tune `microsoft/trocr-base-printed` na polskim zbiorze
[PiotrSty/ocr-pl-lines](https://huggingface.co/datasets/PiotrSty/ocr-pl-lines)
i push wytrenowanego modelu do [PiotrSty/trocr-pl-base](https://huggingface.co/PiotrSty/trocr-pl-base).

**Wymagania:**
- Kaggle: Settings → Accelerator → **GPU T4 x2** (lub P100)
- Kaggle Secrets: `HF_TOKEN` (write) — Add-ons → Secrets
- Internet: ON (Settings → Internet)

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Repo + zaleznosci treningowe (bitsandbytes/peft/jiwer z extras [train])
!git clone https://github.com/PiotrStyla/OCR_engine.git /kaggle/working/OCR_engine
%cd /kaggle/working/OCR_engine
!pip install -q -e ".[train]" 2>&1 | tail -3

In [ ]:
# HF token z Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
print('token ok:', os.environ['HF_TOKEN'][:6] + '...')

In [ ]:
# Pobranie zbioru z HF (train/ 2000, val/ 200)
from huggingface_hub import snapshot_download
data_root = snapshot_download('PiotrSty/ocr-pl-lines', repo_type='dataset')
import os
print('train:', len(os.listdir(f'{data_root}/train')) // 2, 'par')
print('val:  ', len(os.listdir(f'{data_root}/val')) // 2, 'par')

In [ ]:
# TRENING — QLoRA (4-bit + LoRA r=16), domyslne hiperparametry
# ~2000 probek, batch 8 → 250 krokow/epoka; na T4 kilka godzin, na P100 szybciej
!python -m training.train_trocr_pl \
    --train-dir "$data_root/train" \
    --val-dir "$data_root/val" \
    --output /kaggle/working/trocr-pl-base \
    --epochs 3 \
    --batch-size 8

In [ ]:
# Ewaluacja CER/WER na zbiorze walidacyjnym (wytrenowany vs baseline EN)
print('=== wytrenowany PL ===')
!python -m training.evaluate --data "$data_root/val" --model /kaggle/working/trocr-pl-base
print('=== baseline EN ===')
!python -m training.evaluate --data "$data_root/val" --model microsoft/trocr-base-printed

In [ ]:
# Push scalonego modelu na HF Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path='/kaggle/working/trocr-pl-base',
    repo_id='PiotrSty/trocr-pl-base',
)
print('pushed: https://huggingface.co/PiotrSty/trocr-pl-base')

## Notatki
- Sesja Kaggle max ~12h — przy wolniejszym GPU zmniejsz `--epochs` lub uruchom ponownie z checkpointu.
- `/kaggle/working` to jedyny zapiswalny katalog; artefakty mozna pobrac z panelu Output.
- Scalony model laduje sie bezposrednio przez `VisionEncoderDecoderModel.from_pretrained` —
  `OcrConfig(recognizer_pl='PiotrSty/trocr-pl-base')` uzyje go automatycznie.